# Flipkart Gridlock 2.0 — Bengaluru Traffic Demand Prediction
**Objective:** Maximize R² (Score = max(0, 100 × R²))

**Strategy:** CatBoostRegressor with geohash×timestamp target encoding as the dominant signal. Validated using a timestamp-aligned holdout that mirrors the test set distribution.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import lightgbm as lgb

# Reproducibility
SEED = 42
np.random.seed(SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Data

In [2]:
# ── Adjust paths if needed ───────────────────────────────────────────────────
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test.csv'
SUB_PATH    = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'Sample sub  : {sub.shape}')
print('\nTrain columns:', train.columns.tolist())
print('\nTrain dtypes:\n', train.dtypes)
print('\nMissing values (train):\n', train.isnull().sum())
print('\nMissing values (test):\n',  test.isnull().sum())

Train shape : (77299, 11)
Test  shape : (41778, 10)
Sample sub  : (5, 2)

Train columns: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Train dtypes:
 Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object
dtype: object

Missing values (train):
 Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

Missing values (test):
 Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks 

## 3. EDA Snapshot

In [3]:
print('=== TARGET DISTRIBUTION ===')
print(train['demand'].describe())

print('\n=== KEY CARDINALITIES ===')
for col in ['geohash','day','timestamp','RoadType','NumberofLanes','LargeVehicles','Landmarks','Weather']:
    print(f'  {col}: {train[col].nunique()} unique')

# Critical insight: test is day=49 only; train day 49 covers only first ~2h of night
print('\n=== DAY DISTRIBUTION ===')
print(train['day'].value_counts())
print('Test day:', test['day'].unique())

print('\n=== DEMAND BY ROADTYPE ===')
print(train.groupby('RoadType')['demand'].agg(['mean','std','count']).round(4))

# Test timestamp window: 2:15 – 13:45  (47 of 96 possible 15-min slots)
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

test_ts_set = set(test['timestamp'].unique())
print(f'\nTest timestamps ({len(test_ts_set)}): {sorted(test_ts_set, key=ts_to_min)[:5]} … {sorted(test_ts_set, key=ts_to_min)[-3:]}')

# Geohash coverage
train_geo = set(train['geohash'])
test_geo  = set(test['geohash'])
print(f'\nGeohash coverage: {len(test_geo & train_geo)}/{len(test_geo)} test geohashes seen in train')

# geo×ts coverage
train_pairs = set(zip(train['geohash'], train['timestamp']))
test_pairs  = set(zip(test['geohash'],  test['timestamp']))
print(f'geo×timestamp coverage: {len(test_pairs & train_pairs)}/{len(test_pairs)} pairs seen in train')

=== TARGET DISTRIBUTION ===
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64

=== KEY CARDINALITIES ===
  geohash: 1249 unique
  day: 2 unique
  timestamp: 96 unique
  RoadType: 3 unique
  NumberofLanes: 5 unique
  LargeVehicles: 2 unique
  Landmarks: 2 unique
  Weather: 4 unique

=== DAY DISTRIBUTION ===
day
48    69427
49     7872
Name: count, dtype: int64
Test day: [49]

=== DEMAND BY ROADTYPE ===
               mean     std  count
RoadType                          
Highway      0.6108  0.2294   3560
Residential  0.0572  0.0521  69230
Street       0.2732  0.0367   3909

Test timestamps (47): ['2:15', '2:30', '2:45', '3:0', '3:15'] … ['13:15', '13:30', '13:45']

Geohash coverage: 1180/1190 test geohashes seen in train
geo×timestamp coverage: 37136/41778 pairs seen in train


## 4. Validation Strategy

**Design rationale:**
- Test set = day 49, timestamps 2:15–13:45 only.
- Train day 49 covers only timestamps 0:00–2:00 → **not representative** of test.
- Best proxy: day 48 rows **with test-matching timestamps** as the holdout.
- Train fold: day 48 rows outside that window + all of day 49.
- Target encodings computed strictly from the train fold to avoid leakage.

In [4]:
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

TEST_TS_SET = set(test['timestamp'].unique())   # 47 timestamps matching test window

train48 = train[train['day'] == 48].copy()
train49 = train[train['day'] == 49].copy()

val_df = train48[train48['timestamp'].isin(TEST_TS_SET)].copy().reset_index(drop=True)
tr_df  = pd.concat([
    train48[~train48['timestamp'].isin(TEST_TS_SET)],
    train49
], ignore_index=True)

print(f'Train fold : {len(tr_df):,} rows')
print(f'Val fold   : {len(val_df):,} rows  (mirrors test timestamp window)')
print(f'Val geo coverage: {val_df["geohash"].nunique()} / {train["geohash"].nunique()} geohashes')

Train fold : 35,448 rows
Val fold   : 41,851 rows  (mirrors test timestamp window)
Val geo coverage: 1224 / 1249 geohashes


## 5. Feature Engineering

In [5]:
def build_features(df: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all features for df using ref_df as the encoding source.
    For validation: ref_df = tr_df
    For test:       ref_df = full train
    This prevents target leakage.
    """
    df = df.copy()
    global_mean = ref_df['demand'].mean()

    # Missing indicators must be captured before imputation.
    df['temp_missing'] = df['Temperature'].isna().astype(int)
    df['weather_missing'] = df['Weather'].isna().astype(int)
    df['rt_missing'] = df['RoadType'].isna().astype(int)

    # Timestamp features
    df['ts_min'] = df['timestamp'].apply(ts_to_min)
    df['hour'] = df['ts_min'] // 60
    df['minute_slot'] = (df['ts_min'] % 60) // 15
    df['time_slot'] = df['ts_min'] // 15
    df['is_rush_am'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_rush_pm'] = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
    df['is_night'] = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)
    df['sin_hour'] = np.sin(2 * np.pi * df['ts_min'] / 1440)
    df['cos_hour'] = np.cos(2 * np.pi * df['ts_min'] / 1440)
    df['sin_time_slot'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time_slot'] = np.cos(2 * np.pi * df['time_slot'] / 96)

    # Geohash prefix features
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]

    # Road type imputation + ordinal
    rt_order = {'Residential': 0, 'Street': 1, 'Highway': 2}
    geo_rt_mode = (
        ref_df.groupby('geohash')['RoadType']
              .agg(lambda x: x.dropna().mode()[0] if not x.dropna().empty else 'Residential')
              .to_dict()
    )
    df['road_type_filled'] = df['RoadType'].copy()
    rt_na = df['road_type_filled'].isna()
    df.loc[rt_na, 'road_type_filled'] = df.loc[rt_na, 'geohash'].map(geo_rt_mode)
    df['road_type_filled'] = df['road_type_filled'].fillna('Residential')
    df['road_type_ord'] = df['road_type_filled'].map(rt_order).fillna(0).astype(int)

    # Binary flags & interactions
    df['large_veh_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['landmark_bin'] = (df['Landmarks'] == 'Yes').astype(int)
    df['lanes_x_road'] = df['NumberofLanes'] * df['road_type_ord']

    # Temperature imputation
    weather_temp_mean = ref_df.groupby('Weather')['Temperature'].mean().to_dict()
    global_temp = ref_df['Temperature'].mean()
    df['temp_filled'] = df['Temperature'].copy()
    t_na = df['temp_filled'].isna()
    df.loc[t_na, 'temp_filled'] = df.loc[t_na, 'Weather'].map(weather_temp_mean)
    df['temp_filled'] = df['temp_filled'].fillna(global_temp)
    df['weather_filled'] = df['Weather'].fillna('Sunny')

    # Target encodings computed from ref_df only
    ref_enc = ref_df.copy()
    ref_enc['geo4'] = ref_enc['geohash'].str[:4]
    ref_enc['geo5'] = ref_enc['geohash'].str[:5]
    ref_enc['hour'] = ref_enc['timestamp'].apply(ts_to_min) // 60
    ref_enc['time_slot_ref'] = ref_enc['timestamp'].apply(ts_to_min) // 15

    geo_mean = ref_enc.groupby('geohash')['demand'].mean().to_dict()
    geo4_mean = ref_enc.groupby('geo4')['demand'].mean().to_dict()
    geo5_mean = ref_enc.groupby('geo5')['demand'].mean().to_dict()
    geo_ts_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].mean().to_dict()
    geo_hour_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
    rt_ts_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].mean().to_dict()
    rt_hour_dict = ref_enc.groupby(['RoadType', 'hour'])['demand'].mean().to_dict()
    ts_mean_dict = ref_enc.groupby('timestamp')['demand'].mean().to_dict()
    geo_std_dict = ref_enc.groupby('geohash')['demand'].std().fillna(0).to_dict()
    geo_count_dict = ref_enc.groupby('geohash')['demand'].count().to_dict()

    # Count and sum statistics for confidence and smoothed encodings.
    geo_ts_count_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].count().to_dict()
    geo_hour_count_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].count().to_dict()
    rt_ts_count_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].count().to_dict()
    geo_ts_sum_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].sum().to_dict()
    geo_hour_sum_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].sum().to_dict()
    rt_ts_sum_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].sum().to_dict()

    # Frequency encodings from ref_df only.
    geo_freq = ref_enc['geohash'].value_counts(normalize=True).to_dict()
    geo4_freq = ref_enc['geo4'].value_counts(normalize=True).to_dict()
    geo5_freq = ref_enc['geo5'].value_counts(normalize=True).to_dict()

    df['geo_mean'] = df['geohash'].map(geo_mean).fillna(global_mean)
    df['geo4_mean'] = df['geo4'].map(geo4_mean).fillna(global_mean)
    df['geo5_mean'] = df['geo5'].map(geo5_mean).fillna(df['geo4_mean']).fillna(global_mean)

    # geo x timestamp, using the existing leakage-safe methodology.
    df['geo_ts_mean'] = [geo_ts_dict.get((g, ts), np.nan)
                         for g, ts in zip(df['geohash'], df['timestamp'])]
    df['geo_ts_mean'] = df['geo_ts_mean'].fillna(df['geo_mean'])

    # geo x hour, fallback when exact timestamp unseen.
    df['geo_hour_mean'] = [geo_hour_dict.get((g, h), np.nan)
                           for g, h in zip(df['geohash'], df['hour'])]
    df['geo_hour_mean'] = df['geo_hour_mean'].fillna(df['geo_mean'])

    df['rt_ts_mean'] = [rt_ts_dict.get((rt, ts), np.nan)
                        for rt, ts in zip(df['road_type_filled'], df['timestamp'])]
    df['rt_ts_mean'] = df['rt_ts_mean'].fillna(global_mean)

    df['rt_hour_mean'] = [rt_hour_dict.get((rt, h), np.nan)
                          for rt, h in zip(df['road_type_filled'], df['hour'])]
    df['rt_hour_mean'] = df['rt_hour_mean'].fillna(global_mean)

    df['ts_mean'] = df['timestamp'].map(ts_mean_dict).fillna(global_mean)
    df['geo_std'] = df['geohash'].map(geo_std_dict).fillna(0)
    df['geo_count'] = df['geohash'].map(geo_count_dict).fillna(0)
    df['geo_freq'] = df['geohash'].map(geo_freq).fillna(0)
    df['geo4_freq'] = df['geo4'].map(geo4_freq).fillna(0)
    df['geo5_freq'] = df['geo5'].map(geo5_freq).fillna(0)

    df['geo_ts_count'] = [geo_ts_count_dict.get((g, ts), 0)
                          for g, ts in zip(df['geohash'], df['timestamp'])]
    df['geo_hour_count'] = [geo_hour_count_dict.get((g, h), 0)
                            for g, h in zip(df['geohash'], df['hour'])]
    df['rt_ts_count'] = [rt_ts_count_dict.get((rt, ts), 0)
                         for rt, ts in zip(df['road_type_filled'], df['timestamp'])]

    df['geo_ts_seen'] = (df['geo_ts_count'] > 0).astype(int)
    df['geo_hour_seen'] = (df['geo_hour_count'] > 0).astype(int)
    df['rt_ts_seen'] = (df['rt_ts_count'] > 0).astype(int)

    for smooth_m in [5, 10, 20, 50]:
        geo_ts_sum = np.array([geo_ts_sum_dict.get((g, ts), 0.0)
                               for g, ts in zip(df['geohash'], df['timestamp'])])
        geo_hour_sum = np.array([geo_hour_sum_dict.get((g, h), 0.0)
                                 for g, h in zip(df['geohash'], df['hour'])])
        rt_ts_sum = np.array([rt_ts_sum_dict.get((rt, ts), 0.0)
                              for rt, ts in zip(df['road_type_filled'], df['timestamp'])])

        df[f'geo_ts_smooth_m{smooth_m}'] = (
            geo_ts_sum + global_mean * smooth_m
        ) / (df['geo_ts_count'].to_numpy() + smooth_m)
        df[f'geo_hour_smooth_m{smooth_m}'] = (
            geo_hour_sum + global_mean * smooth_m
        ) / (df['geo_hour_count'].to_numpy() + smooth_m)
        df[f'rt_ts_smooth_m{smooth_m}'] = (
            rt_ts_sum + global_mean * smooth_m
        ) / (df['rt_ts_count'].to_numpy() + smooth_m)

    df['geo_ts_confidence'] = df['geo_ts_mean'] * np.log1p(df['geo_ts_count'])
    df['geo_hour_confidence'] = df['geo_hour_mean'] * np.log1p(df['geo_hour_count'])
    df['rt_ts_confidence'] = df['rt_ts_mean'] * np.log1p(df['rt_ts_count'])

    # ── NEW FEATURE 1: time_slot_demand_mean ─────────────────────────────────────
    # Mean demand by 15-minute time slot, computed from ref_df only (no leakage).
    ts_slot_demand_dict = (
        ref_enc.groupby('time_slot_ref')['demand']
        .mean()
        .to_dict()
    )
    df['time_slot_demand_mean'] = (
        df['time_slot'].map(ts_slot_demand_dict).fillna(global_mean)
    )

    # ── NEW FEATURE 2: prev_day_ts_demand ────────────────────────────────────────
    # For row (day=d, geohash=g, time_slot=s), look up mean demand for
    # (day=d-1, geohash=g, time_slot=s) from ref_df. Leak-free because d-1 < d.
    # For test rows (day=49), this retrieves day-48 demand — the highest-coverage
    # exact signal available.
    prev_day_lookup = (
        ref_enc.groupby(['day', 'geohash', 'time_slot_ref'])['demand']
        .mean()
        .to_dict()
    )
    df['prev_day_ts_demand'] = [
        prev_day_lookup.get((int(d) - 1, g, int(ts)), np.nan)
        for d, g, ts in zip(df['day'], df['geohash'], df['time_slot'])
    ]
    # Fallback chain: geo_mean (already computed above) → global_mean
    df['prev_day_ts_demand'] = (
        df['prev_day_ts_demand']
        .fillna(df['geo_mean'])
        .fillna(global_mean)
    )

    # ── NEW FEATURE 3: demand_ratio_gh_hour ─────────────────────────────────────
    # Ratio of geo×hour mean demand to geohash-level mean demand.
    # Captures time-of-day demand pattern relative to the location baseline.
    # geo_hour_mean and geo_mean are already computed above.
    df['demand_ratio_gh_hour'] = df['geo_hour_mean'] / (df['geo_mean'] + 1e-8)

    df['geo_ts_delta'] = df['geo_ts_mean'] - df['geo_mean']

    return df


print('Feature engineering function defined.')

Feature engineering function defined.


In [6]:
# ════════════════════════════════════════════════════════════════════════
# COVERAGE & FEATURE DIAGNOSTICS (v6 new features)
# ════════════════════════════════════════════════════════════════════════
print('Building fold features for diagnostics...')
tr_fe  = build_features(tr_df, tr_df)
val_fe = build_features(val_df, tr_df)
test_fe_diag = build_features(test, train)

y_tr  = tr_fe['demand']
y_val = val_fe['demand']

print()
print('=== prev_day_ts_demand COVERAGE ===')

# Re-derive the raw (pre-fillna) lookup to compute TRUE hit rates
def _build_prev_lookup(ref):
    ref2 = ref.copy()
    ref2['time_slot_ref'] = ref2['timestamp'].apply(ts_to_min) // 15
    return (
        ref2.groupby(['day', 'geohash', 'time_slot_ref'])['demand']
        .mean()
        .to_dict()
    )

def _raw_prev(df_in, lookup):
    return pd.Series([
        lookup.get((int(d) - 1, g, int(ts)), np.nan)
        for d, g, ts in zip(df_in['day'], df_in['geohash'], df_in['time_slot'])
    ], index=df_in.index)

train_lookup = _build_prev_lookup(tr_df)
full_lookup  = _build_prev_lookup(train)

raw_train = _raw_prev(tr_fe,       train_lookup)
raw_val   = _raw_prev(val_fe,      train_lookup)
raw_test  = _raw_prev(test_fe_diag, full_lookup)

for split, raw in [('Train', raw_train), ('Val', raw_val), ('Test', raw_test)]:
    n_total = len(raw)
    n_hit   = raw.notna().sum()
    print(f'  {split:<6}: coverage={n_hit/n_total:.3%}  ({n_hit:,}/{n_total:,})  missing={raw.isna().mean():.3%}')

# Correlation and standalone R² for prev_day_ts_demand
corr_prev = tr_fe['prev_day_ts_demand'].corr(y_tr)
r2_prev_standalone = r2_score(y_val, val_fe['prev_day_ts_demand'])
print(f'\n  Correlation with demand (train fold) : {corr_prev:.6f}')
print(f'  Standalone Val R²                    : {r2_prev_standalone:.6f}')

print()
print('=== demand_ratio_gh_hour DIAGNOSTICS ===')
print(val_fe['demand_ratio_gh_hour'].describe().round(6).to_string())

print()
print('=== time_slot_demand_mean DIAGNOSTICS ===')
print(f'  Unique time slots (train fold): {tr_fe["time_slot"].nunique()}')
print(f'  Val coverage                  : {val_fe["time_slot_demand_mean"].notna().mean():.3%}')
r2_ts_standalone = r2_score(y_val, val_fe['time_slot_demand_mean'])
print(f'  Standalone Val R²             : {r2_ts_standalone:.6f}')

Building fold features for diagnostics...



=== prev_day_ts_demand COVERAGE ===
  Train : coverage=18.119%  (6,423/35,448)  missing=81.881%
  Val   : coverage=0.000%  (0/41,851)  missing=100.000%
  Test  : coverage=88.889%  (37,136/41,778)  missing=11.111%

  Correlation with demand (train fold) : 0.760915
  Standalone Val R²                    : 0.569558

=== demand_ratio_gh_hour DIAGNOSTICS ===
count    41851.000000
mean         1.024258
std          0.171018
min          0.011494
25%          1.000000
50%          1.000000
75%          1.000000
max          5.164194

=== time_slot_demand_mean DIAGNOSTICS ===
  Unique time slots (train fold): 49
  Val coverage                  : 100.000%
  Standalone Val R²             : -0.028544


## 6. Train CatBoost — Validation Fold

In [7]:
print('Building train fold features...')
tr_fe = build_features(tr_df, tr_df)
print('Building val fold features...')
val_fe = build_features(val_df, tr_df)
print('Done. Shape:', tr_fe.shape)

BASE_FEATURES = [
    # Time
    'ts_min', 'hour', 'minute_slot', 'is_rush_am', 'is_rush_pm', 'is_night',
    'sin_hour', 'cos_hour',
    # Road / infrastructure
    'NumberofLanes', 'large_veh_bin', 'landmark_bin', 'lanes_x_road',
    'temp_filled', 'road_type_ord',
    # Target encodings
    'geo_mean', 'geo4_mean', 'geo_ts_mean', 'geo_hour_mean',
    'rt_ts_mean', 'rt_hour_mean', 'ts_mean', 'geo_std', 'geo_count',
    'geo_ts_delta',
    # Helpful v2 features retained
    'geo_freq', 'geo4_freq', 'geo5_freq',
    'temp_missing', 'weather_missing', 'rt_missing',
    # NEW v6 features
    'time_slot_demand_mean',
    'prev_day_ts_demand',
    'demand_ratio_gh_hour',
    # Categoricals handled natively by CatBoost
    'geohash', 'geo4', 'road_type_filled', 'weather_filled'
]

BASE_CAT_FEATURES = ['geohash', 'geo4', 'road_type_filled', 'weather_filled']

COUNT_FEATURES = ['geo_ts_count', 'geo_hour_count', 'rt_ts_count']  # geo_count already exists in BASE_FEATURES
SEEN_FEATURES = ['geo_ts_seen', 'geo_hour_seen', 'rt_ts_seen']
SMOOTH_MS = [5, 10, 20, 50]
CONFIDENCE_FEATURES = ['geo_ts_confidence', 'geo_hour_confidence', 'rt_ts_confidence']


def add_unique(base, additions):
    result = list(base)
    for item in additions:
        if item not in result:
            result.append(item)
    return result


active_features = BASE_FEATURES.copy()
active_cat_features = BASE_CAT_FEATURES.copy()

y_tr = tr_fe['demand']
y_val = val_fe['demand']

print(f'Training on {len(tr_fe):,} rows, validating on {len(val_fe):,} rows.')
print(f'Current v2 feature set: {len(active_features)} ({len(active_cat_features)} categorical)')

Building train fold features...


Building val fold features...


Done. Shape: (35448, 72)
Training on 35,448 rows, validating on 41,851 rows.
Current v2 feature set: 37 (4 categorical)


In [8]:
CAT_BASE_PARAMS = dict(
    iterations=5000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=7,
    min_data_in_leaf=20,
    bagging_temperature=1.0,
    random_strength=1.0,
    eval_metric='R2',
    loss_function='RMSE',
    random_seed=SEED,
    early_stopping_rounds=200,
)


def train_catboost_eval(features, cat_features, param_overrides=None, verbose=False):
    params = CAT_BASE_PARAMS.copy()
    if param_overrides:
        params.update(param_overrides)

    model = CatBoostRegressor(
        **params,
        cat_features=[c for c in cat_features if c in features],
        verbose=verbose,
    )
    model.fit(
        tr_fe[features], y_tr,
        eval_set=(val_fe[features], y_val),
        use_best_model=True,
    )
    preds = model.predict(val_fe[features])
    score = r2_score(y_val, preds)
    return score, model, preds


# Phase 1 diagnostics
raw_diagnostics = {
    'geo_ts_mean': r2_score(y_val, val_fe['geo_ts_mean']),
    'geo_hour_mean': r2_score(y_val, val_fe['geo_hour_mean']),
    'geo_mean': r2_score(y_val, val_fe['geo_mean']),
    'rt_ts_mean': r2_score(y_val, val_fe['rt_ts_mean']),
    'ts_mean': r2_score(y_val, val_fe['ts_mean']),
}
coverage = {
    'geo_ts_hit_rate': val_fe['geo_ts_seen'].mean(),
    'geo_ts_hits': int(val_fe['geo_ts_seen'].sum()),
    'geo_hour_hit_rate': val_fe['geo_hour_seen'].mean(),
    'geo_hour_hits': int(val_fe['geo_hour_seen'].sum()),
    'validation_rows': len(val_fe),
}

print('=== PHASE 1 DIAGNOSTICS ===')
for feature_name, score in raw_diagnostics.items():
    print(f'{feature_name:<14} raw R2: {score:.6f}')
print(f"geo x timestamp coverage: {coverage['geo_ts_hits']:,}/{coverage['validation_rows']:,} = {coverage['geo_ts_hit_rate']:.3%}")
print(f"geo x hour coverage     : {coverage['geo_hour_hits']:,}/{coverage['validation_rows']:,} = {coverage['geo_hour_hit_rate']:.3%}")
print('Interpretation: geo_ts_mean has no exact validation support and falls back to geo_mean; geo_hour_mean has limited support and is the strongest raw encoding.')

print()
print('Training current v2 CatBoost baseline...')
current_r2, current_model, current_preds = train_catboost_eval(active_features, active_cat_features)
print(f'Current feature-set CatBoost R2: {current_r2:.6f}')

feature_eval_rows = []

def evaluate_group(group_name, candidate_features, candidate_cat_features=None):
    global active_features, active_cat_features, current_r2, current_model, current_preds
    if candidate_cat_features is None:
        candidate_cat_features = active_cat_features

    before = current_r2
    after, candidate_model, candidate_preds = train_catboost_eval(candidate_features, candidate_cat_features)
    gain = after - before
    keep = gain > 0

    if keep:
        active_features = candidate_features
        active_cat_features = candidate_cat_features
        current_r2 = after
        current_model = candidate_model
        current_preds = candidate_preds

    feature_eval_rows.append({
        'Experiment': group_name,
        'Validation R2 Before': before,
        'Validation R2 After': after,
        'Gain': gain,
        'Kept': keep,
    })

    print()
    print(f'{group_name}')
    print(f'Validation R2 Before: {before:.6f}')
    print(f'Validation R2 After : {after:.6f}')
    print(f'Gain                : {gain:+.6f}')
    print('Decision            :', 'kept' if keep else 'removed from final feature set')
    return after, gain, keep


# Phase 2: count features
candidate_features = add_unique(active_features, COUNT_FEATURES)
evaluate_group('count_features', candidate_features)
print('Note: geo_count was already present in the current v2 feature set; new count candidates are geo_ts_count, geo_hour_count, rt_ts_count.')

# Phase 3: seen/unseen indicators
candidate_features = add_unique(active_features, SEEN_FEATURES)
evaluate_group('seen_unseen_features', candidate_features)

# Phase 4: smoothed target encodings
smooth_rows = []
smooth_baseline = current_r2
best_smooth = {'M': None, 'score': -np.inf, 'features': None}
for smooth_m in SMOOTH_MS:
    smooth_features = [f'geo_ts_smooth_m{smooth_m}', f'geo_hour_smooth_m{smooth_m}', f'rt_ts_smooth_m{smooth_m}']
    candidate_features = add_unique(active_features, smooth_features)
    score, _, _ = train_catboost_eval(candidate_features, active_cat_features)
    smooth_rows.append({'M': smooth_m, 'Validation R2': score, 'Gain vs Before': score - smooth_baseline})
    print(f'Smoothing M={smooth_m:<2} -> R2={score:.6f} gain={score - smooth_baseline:+.6f}')
    if score > best_smooth['score']:
        best_smooth = {'M': smooth_m, 'score': score, 'features': smooth_features}

if best_smooth['score'] > current_r2:
    before = current_r2
    active_features = add_unique(active_features, best_smooth['features'])
    current_r2, current_model, current_preds = train_catboost_eval(active_features, active_cat_features)
    feature_eval_rows.append({
        'Experiment': f"smoothed_encodings_m{best_smooth['M']}",
        'Validation R2 Before': before,
        'Validation R2 After': current_r2,
        'Gain': current_r2 - before,
        'Kept': True,
    })
    print(f"Best smoothing M={best_smooth['M']} kept with R2={current_r2:.6f}")
else:
    feature_eval_rows.append({
        'Experiment': 'smoothed_encodings',
        'Validation R2 Before': current_r2,
        'Validation R2 After': best_smooth['score'],
        'Gain': best_smooth['score'] - current_r2,
        'Kept': False,
    })
    print(f"Best smoothing M={best_smooth['M']} did not improve and was removed.")

smooth_df = pd.DataFrame(smooth_rows)

# Phase 5: confidence-weighted features, evaluated one at a time.
for confidence_feature in CONFIDENCE_FEATURES:
    candidate_features = add_unique(active_features, [confidence_feature])
    evaluate_group(confidence_feature, candidate_features)

feature_eval_df = pd.DataFrame(feature_eval_rows)
ranked_improvements = feature_eval_df.sort_values('Gain', ascending=False).reset_index(drop=True)

print()
print('Selected v3 feature additions:', feature_eval_df.loc[feature_eval_df['Kept'], 'Experiment'].tolist())
print(f'Selected feature count: {len(active_features)} ({len(active_cat_features)} categorical)')

# Phase 6: micro hyperparameter search on selected features.
print()
print('=== PHASE 6 MICRO HYPERPARAMETER SEARCH ===')
tuning_rows = []
best_cat_r2 = current_r2
best_cat_model = current_model
best_cat_preds = current_preds
best_cat_params = {'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05}

for learning_rate in [0.03, 0.05]:
    for depth in [5, 6, 7]:
        for l2_leaf_reg in [5, 7, 10]:
            overrides = {'learning_rate': learning_rate, 'depth': depth, 'l2_leaf_reg': l2_leaf_reg}
            if (learning_rate == 0.05 and depth == 6 and l2_leaf_reg == 7):
                score, model, preds = current_r2, current_model, current_preds
            else:
                score, model, preds = train_catboost_eval(active_features, active_cat_features, overrides)

            tuning_rows.append({
                'learning_rate': learning_rate,
                'depth': depth,
                'l2_leaf_reg': l2_leaf_reg,
                'Validation R2': score,
            })
            print(f'lr={learning_rate:<4} depth={depth} l2={l2_leaf_reg:<2} -> R2={score:.6f}')

            if score > best_cat_r2:
                best_cat_r2 = score
                best_cat_model = model
                best_cat_preds = preds
                best_cat_params = overrides

cat_tuning_df = pd.DataFrame(tuning_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
print()
print('Best CatBoost params:', best_cat_params)
print(f'Best CatBoost R2: {best_cat_r2:.6f}')

# Phase 7: CatBoost ensemble.
print()
print('=== PHASE 7 CATBOOST ENSEMBLE ===')
ensemble_specs = {
    'A': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 5},
    'B': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 7},
    'C': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 10},
}
ensemble_models = {}
ensemble_preds = {}
ensemble_single_rows = []
for name, params in ensemble_specs.items():
    score, model, preds = train_catboost_eval(active_features, active_cat_features, params)
    ensemble_models[name] = model
    ensemble_preds[name] = preds
    ensemble_single_rows.append({'Model': name, **params, 'Validation R2': score})
    print(f"Model {name} {params} -> R2={score:.6f}")

blend_rows = []
weight_grid = np.round(np.arange(0.05, 1.00, 0.05), 2)
for combo in [('A', 'B'), ('A', 'C'), ('B', 'C')]:
    best_combo = {'score': -np.inf, 'weights': None}
    for w in weight_grid:
        preds = w * ensemble_preds[combo[0]] + (1 - w) * ensemble_preds[combo[1]]
        score = r2_score(y_val, preds)
        if score > best_combo['score']:
            best_combo = {'score': score, 'weights': {combo[0]: float(w), combo[1]: float(1 - w)}}
    blend_rows.append({'Blend': '+'.join(combo), 'Weights': best_combo['weights'], 'Validation R2': best_combo['score']})
    print(f"Blend {'+'.join(combo)} best weights {best_combo['weights']} -> R2={best_combo['score']:.6f}")

best_combo = {'score': -np.inf, 'weights': None}
for w_a in weight_grid:
    for w_b in weight_grid:
        w_c = round(1 - w_a - w_b, 2)
        if w_c < 0.05:
            continue
        preds = w_a * ensemble_preds['A'] + w_b * ensemble_preds['B'] + w_c * ensemble_preds['C']
        score = r2_score(y_val, preds)
        if score > best_combo['score']:
            best_combo = {'score': score, 'weights': {'A': float(w_a), 'B': float(w_b), 'C': float(w_c)}}
blend_rows.append({'Blend': 'A+B+C', 'Weights': best_combo['weights'], 'Validation R2': best_combo['score']})
print(f"Blend A+B+C best weights {best_combo['weights']} -> R2={best_combo['score']:.6f}")

ensemble_single_df = pd.DataFrame(ensemble_single_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
ensemble_df = pd.DataFrame(blend_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
best_ensemble = ensemble_df.iloc[0].to_dict()
best_ensemble_r2 = float(best_ensemble['Validation R2'])

if best_ensemble_r2 > best_cat_r2:
    final_strategy = 'ensemble'
    best_validation_r2 = best_ensemble_r2
else:
    final_strategy = 'single_catboost'
    best_validation_r2 = best_cat_r2

print()
print('Best ensemble:', best_ensemble)
print(f'Best validation R2 selected for final output: {best_validation_r2:.6f} ({final_strategy})')

final_features = active_features.copy()
final_cat_features = active_cat_features.copy()

=== PHASE 1 DIAGNOSTICS ===
geo_ts_mean    raw R2: 0.569558
geo_hour_mean  raw R2: 0.574106
geo_mean       raw R2: 0.569558
rt_ts_mean     raw R2: -0.028544
ts_mean        raw R2: -0.028544
geo x timestamp coverage: 0/41,851 = 0.000%
geo x hour coverage     : 2,556/41,851 = 6.107%
Interpretation: geo_ts_mean has no exact validation support and falls back to geo_mean; geo_hour_mean has limited support and is the strongest raw encoding.

Training current v2 CatBoost baseline...


Current feature-set CatBoost R2: 0.549742



count_features
Validation R2 Before: 0.549742
Validation R2 After : 0.578268
Gain                : +0.028526
Decision            : kept
Note: geo_count was already present in the current v2 feature set; new count candidates are geo_ts_count, geo_hour_count, rt_ts_count.



seen_unseen_features
Validation R2 Before: 0.578268
Validation R2 After : 0.578268
Gain                : +0.000000
Decision            : removed from final feature set


Smoothing M=5  -> R2=0.401466 gain=-0.176802


...

lr=0.05 depth=7 l2=7  -> R2=0.599075


lr=0.05 depth=7 l2=10 -> R2=0.603318

Best CatBoost params: {'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 5}
Best CatBoost R2: 0.611772

=== PHASE 7 CATBOOST ENSEMBLE ===


Model A {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 5} -> R2=0.605034


Model B {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 7} -> R2=0.603607


Model C {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 10} -> R2=0.607880
Blend A+B best weights {'A': 0.95, 'B': 0.050000000000000044} -> R2=0.604971
Blend A+C best weights {'A': 0.05, 'C': 0.95} -> R2=0.607747
Blend B+C best weights {'B': 0.05, 'C': 0.95} -> R2=0.607679
Blend A+B+C best weights {'A': 0.05, 'B': 0.05, 'C': 0.9} -> R2=0.607545

Best ensemble: {'Blend': 'A+C', 'Weights': {'A': 0.05, 'C': 0.95}, 'Validation R2': 0.6077465504466582}
Best validation R2 selected for final output: 0.611772 (single_catboost)


## 7. Validation Score & Feature Importance

In [9]:
# Phase 8: feature importance analysis.
if final_strategy == 'ensemble':
    final_weights = best_ensemble['Weights']
    importance_parts = []
    for model_name, weight in final_weights.items():
        importance_parts.append(
            pd.Series(ensemble_models[model_name].get_feature_importance(), index=final_features) * weight
        )
    fi = sum(importance_parts).sort_values(ascending=False)
else:
    fi = pd.Series(
        best_cat_model.get_feature_importance(),
        index=final_features,
    ).sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (top 30, final validation strategy) ===')
print(fi.head(30).round(4).to_string())

inspect_features = [
    'geo_ts_mean', 'geo_ts_count', 'geo_ts_seen',
    'geo_hour_mean', 'geo_hour_count', 'geo_hour_seen',
    'rt_ts_mean', 'rt_ts_count', 'rt_ts_seen',
    'geo_ts_smooth_m5', 'geo_ts_smooth_m10', 'geo_ts_smooth_m20', 'geo_ts_smooth_m50',
    'geo_hour_smooth_m5', 'geo_hour_smooth_m10', 'geo_hour_smooth_m20', 'geo_hour_smooth_m50',
    'geo_freq', 'geo4_freq', 'geo5_freq',
]
importance_review = pd.DataFrame({
    'Feature': inspect_features,
    'In Final Features': [f in final_features for f in inspect_features],
    'Importance': [float(fi.get(f, 0.0)) for f in inspect_features],
}).sort_values(['In Final Features', 'Importance'], ascending=[False, False])

print()
print('=== TARGETED IMPORTANCE REVIEW ===')
print(importance_review.to_string(index=False))

print()
print('=== FEATURE EXPERIMENT RESULTS ===')
print(feature_eval_df.assign(
    **{
        'Validation R2 Before': feature_eval_df['Validation R2 Before'].round(6),
        'Validation R2 After': feature_eval_df['Validation R2 After'].round(6),
        'Gain': feature_eval_df['Gain'].round(6),
    }
).to_string(index=False))

print()
print('=== SMOOTHING RESULTS ===')
print(smooth_df.assign(
    **{
        'Validation R2': smooth_df['Validation R2'].round(6),
        'Gain vs Before': smooth_df['Gain vs Before'].round(6),
    }
).to_string(index=False))

print()
print('=== HYPERPARAMETER RESULTS ===')
print(cat_tuning_df.assign(**{'Validation R2': cat_tuning_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== ENSEMBLE SINGLE MODELS ===')
print(ensemble_single_df.assign(**{'Validation R2': ensemble_single_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== ENSEMBLE BLENDS ===')
print(ensemble_df.assign(**{'Validation R2': ensemble_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== FINAL FEATURE LIST ===')
print(pd.Series(final_features).to_string(index=False))

=== FEATURE IMPORTANCE (top 30, final validation strategy) ===
geo_ts_mean              47.7067
road_type_filled         12.8157
geo_hour_mean            11.5556
road_type_ord             4.3508
lanes_x_road              4.3086
rt_ts_count               3.8050
prev_day_ts_demand        2.3548
rt_ts_confidence          2.0056
NumberofLanes             1.9180
rt_ts_mean                1.8348
large_veh_bin             1.4810
rt_hour_mean              1.2450
geo_mean                  0.9743
geo_ts_delta              0.6747
geo_hour_count            0.5439
hour                      0.4680
ts_min                    0.3937
geo5_freq                 0.3491
geo_count                 0.2328
geo_ts_count              0.2325
geo_std                   0.1820
geo4_mean                 0.1300
demand_ratio_gh_hour      0.1087
rt_missing                0.0983
geo4                      0.0775
time_slot_demand_mean     0.0535
weather_missing           0.0299
sin_hour                  0.0262
cos_hour     

In [10]:
print('=== RANKED FEATURE ADDITIONS ===')
print(ranked_improvements.assign(
    **{
        'Validation R2 Before': ranked_improvements['Validation R2 Before'].round(6),
        'Validation R2 After': ranked_improvements['Validation R2 After'].round(6),
        'Gain': ranked_improvements['Gain'].round(6),
    }
).to_string(index=False))

print()
print('Features kept from v3 experiments:')
print(feature_eval_df.loc[feature_eval_df['Kept'], 'Experiment'].to_string(index=False))

=== RANKED FEATURE ADDITIONS ===
          Experiment  Validation R2 Before  Validation R2 After      Gain  Kept
      count_features              0.549742             0.578268  0.028526  True
    rt_ts_confidence              0.578268             0.603607  0.025339  True
seen_unseen_features              0.578268             0.578268  0.000000 False
 geo_hour_confidence              0.578268             0.532242 -0.046027 False
  smoothed_encodings              0.578268             0.432226 -0.146042 False
   geo_ts_confidence              0.578268             0.337328 -0.240940 False

Features kept from v3 experiments:
  count_features
rt_ts_confidence


In [11]:
# ════════════════════════════════════════════════════════════════════════
# V6 EXPERIMENT TRACKING — Incremental Feature Evaluation
# ════════════════════════════════════════════════════════════════════════
experiment_log = []   # Will be printed as the final experiment table

# Snapshot the current best from the existing pipeline
_current_features  = list(final_features)
_current_cat       = list(final_cat_features)
_baseline_r2       = best_cat_r2

print('=== V6 EXPERIMENT TRACKING ===')
print(f'Baseline (post-v2-experiments) Val R2: {_baseline_r2:.6f}')
print()

def _try_feature(label, new_feats, cat_feats_add=None, coverage_note=''):
    """Add candidate features on top of current best set; update if gain > 0."""
    global _current_features, _current_cat, _baseline_r2

    cand_feats = _current_features + [f for f in new_feats if f not in _current_features]
    cand_cat   = _current_cat   + ([c for c in cat_feats_add if c not in _current_cat]
                                    if cat_feats_add else [])

    r2_before = _baseline_r2
    r2_after, model_after, preds_after = train_catboost_eval(cand_feats, cand_cat)
    gain = r2_after - r2_before
    keep = gain > 0

    if keep:
        _current_features = cand_feats
        _current_cat      = cand_cat
        _baseline_r2      = r2_after
        status = 'YES'
        tag    = '[KEPT   ]'
    else:
        status = 'NO'
        tag    = '[DROPPED]'

    print(f'{tag} {label:<35}  R2: {r2_before:.6f} → {r2_after:.6f}  ({gain:+.6f})')

    experiment_log.append({
        'Feature Added'        : label,
        'Coverage'             : coverage_note,
        'Validation R2 Before' : round(r2_before, 6),
        'Validation R2 After'  : round(r2_after,  6),
        'Gain'                 : round(gain, 6),
        'Keep?'                : status,
    })
    return keep, r2_after, model_after, preds_after


# Compute coverage strings for the table
_cov_prev  = (f"Val={raw_val.notna().mean():.1%}  "
              f"Test={raw_test.notna().mean():.1%}")
_cov_ratio = "100% (computed from existing features)"
_cov_ts    = "100% (all 96 slots in training)"

keep_prev,  r2_exp1, model_exp1, preds_exp1 = _try_feature(
    'prev_day_ts_demand',    ['prev_day_ts_demand'],    coverage_note=_cov_prev)

keep_ratio, r2_exp2, model_exp2, preds_exp2 = _try_feature(
    'demand_ratio_gh_hour',  ['demand_ratio_gh_hour'],  coverage_note=_cov_ratio)

keep_ts,    r2_exp3, model_exp3, preds_exp3 = _try_feature(
    'time_slot_demand_mean', ['time_slot_demand_mean'], coverage_note=_cov_ts)

print()
print(f'Final feature count after v6 experiments: {len(_current_features)}')

# Overwrite final_features for downstream cells
final_features_v6    = list(_current_features)
final_cat_features_v6 = list(_current_cat)
best_r2_v6           = _baseline_r2

=== V6 EXPERIMENT TRACKING ===
Baseline (post-v2-experiments) Val R2: 0.611772



[DROPPED] prev_day_ts_demand                   R2: 0.611772 → 0.603607  (-0.008165)


[DROPPED] demand_ratio_gh_hour                 R2: 0.611772 → 0.603607  (-0.008165)


[DROPPED] time_slot_demand_mean                R2: 0.611772 → 0.603607  (-0.008165)

Final feature count after v6 experiments: 41


In [12]:
# ════════════════════════════════════════════════════════════════════════
# LIGHTGBM — Same Feature Set as Best CatBoost v6
# ════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import LabelEncoder as _LE

print('=== LIGHTGBM TRAINING ===')

_LGB_PARAMS = {
    'objective'       : 'regression',
    'metric'          : 'rmse',
    'boosting_type'   : 'gbdt',
    'learning_rate'   : 0.05,
    'num_leaves'      : 255,
    'max_depth'       : -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq'    : 5,
    'reg_alpha'       : 0.1,
    'reg_lambda'      : 5.0,
    'cat_smooth'      : 10,
    'n_jobs'          : -1,
    'random_state'    : 42,
    'verbosity'       : -1,
}

_CAT_COLS_LGB = ['geohash', 'geo4', 'road_type_filled', 'weather_filled']

def _encode_for_lgb(tr_data, val_data, feature_list):
    """Label-encode categorical columns; return encoded copies + cat indices."""
    tr_enc  = tr_data[feature_list].copy()
    val_enc = val_data[feature_list].copy()
    cat_idxs = []
    encoders = {}
    for col in _CAT_COLS_LGB:
        if col not in feature_list:
            continue
        le = _LE()
        combined = pd.concat([tr_enc[col], val_enc[col]], ignore_index=True)
        le.fit(combined.astype(str))
        tr_enc[col]  = le.transform(tr_enc[col].astype(str))
        val_enc[col] = le.transform(
            val_enc[col].astype(str).map(
                lambda x, lec=le: x if x in lec.classes_ else lec.classes_[0]
            )
        )
        cat_idxs.append(feature_list.index(col))
        encoders[col] = le
    return tr_enc, val_enc, cat_idxs, encoders

tr_lgb, val_lgb, cat_idxs_lgb, lgb_encoders = _encode_for_lgb(
    tr_fe, val_fe, final_features_v6
)

dtrain_lgb = lgb.Dataset(
    tr_lgb[final_features_v6], label=y_tr,
    categorical_feature=cat_idxs_lgb or 'auto',
)
dval_lgb = lgb.Dataset(
    val_lgb[final_features_v6], label=y_val,
    reference=dtrain_lgb,
)

lgb_model_val = lgb.train(
    _LGB_PARAMS,
    dtrain_lgb,
    num_boost_round=5000,
    valid_sets=[dval_lgb],
    callbacks=[
        lgb.early_stopping(200, verbose=False),
        lgb.log_evaluation(0),
    ],
)

lgb_preds_val = lgb_model_val.predict(val_lgb[final_features_v6])
lgb_r2_val    = r2_score(y_val, lgb_preds_val)
print(f'LightGBM Val R2  : {lgb_r2_val:.6f}')
print(f'Best iteration   : {lgb_model_val.best_iteration}')

# ── Get the CatBoost validation predictions from best v6 model ───────────────
# Re-train (or fetch) to get validation preds aligned with final_features_v6
cat_r2_v6, cat_model_v6, cat_preds_v6 = train_catboost_eval(
    final_features_v6, final_cat_features_v6
)
print(f'CatBoost v6 Val R2: {cat_r2_v6:.6f}')

# ── Blend Search ─────────────────────────────────────────────────────────────
print()
print('=== BLEND SEARCH (alpha = CatBoost weight) ===')
best_blend_r2 = -np.inf
best_alpha    = 0.5

for alpha in np.round(np.arange(0.0, 1.01, 0.05), 2):
    blend_r2 = r2_score(y_val, alpha * cat_preds_v6 + (1 - alpha) * lgb_preds_val)
    if blend_r2 > best_blend_r2:
        best_blend_r2 = blend_r2
        best_alpha    = alpha

print(f'CatBoost only   R2: {cat_r2_v6:.6f}')
print(f'LightGBM only   R2: {lgb_r2_val:.6f}')
print(f'Best alpha (CB) : {best_alpha:.2f}')
print(f'Best blend      R2: {best_blend_r2:.6f}')

best_blend_val_preds = best_alpha * cat_preds_v6 + (1 - best_alpha) * lgb_preds_val

# Append to experiment log
experiment_log.append({
    'Feature Added'        : f'LightGBM blend (CB_weight={best_alpha:.2f})',
    'Coverage'             : 'N/A',
    'Validation R2 Before' : round(cat_r2_v6,    6),
    'Validation R2 After'  : round(best_blend_r2, 6),
    'Gain'                 : round(best_blend_r2 - cat_r2_v6, 6),
    'Keep?'                : 'YES' if best_blend_r2 > cat_r2_v6 else 'NO',
})

=== LIGHTGBM TRAINING ===


LightGBM Val R2  : 0.617472
Best iteration   : 143


CatBoost v6 Val R2: 0.603607

=== BLEND SEARCH (alpha = CatBoost weight) ===
CatBoost only   R2: 0.603607
LightGBM only   R2: 0.617472
Best alpha (CB) : 0.00
Best blend      R2: 0.617472


In [13]:
# ════════════════════════════════════════════════════════════════════════
# FEATURE IMPORTANCE REPORT
# ════════════════════════════════════════════════════════════════════════
print('=== TOP 50 FEATURES (CatBoost v6 validation model) ===')
fi_series = pd.Series(
    cat_model_v6.get_feature_importance(),
    index=final_features_v6,
).sort_values(ascending=False)

print(fi_series.head(50).round(4).to_string())

# Required specific feature report
_REQUIRED_FEATS = [
    'prev_day_ts_demand',
    'geo_ts_mean',
    'geo_hour_mean',
    'demand_ratio_gh_hour',
    'time_slot_demand_mean',
    'geo_freq',
    'geo4_freq',
    'geo5_freq',
]
_ROAD_FEATS = [f for f in fi_series.index
               if any(kw in f.lower() for kw in ['road', 'rt_', 'roadtype'])]

print()
print('=== SPECIFIC FEATURE IMPORTANCES ===')
print(f'{"Feature":<40} {"Importance":>12} {"Rank":>6} {"In Model":>10}')
print('-' * 72)
for feat in _REQUIRED_FEATS + _ROAD_FEATS:
    imp  = fi_series.get(feat, 0.0)
    rank = int((fi_series > imp).sum()) + 1 if imp > 0 else 'N/A'
    in_m = feat in final_features_v6
    print(f'{feat:<40} {imp:>12.4f} {str(rank):>6} {str(in_m):>10}')

print()
print('=== FINAL EXPERIMENT TRACKING TABLE ===')
exp_df = pd.DataFrame(experiment_log)
print(exp_df.to_string(index=False))

=== TOP 50 FEATURES (CatBoost v6 validation model) ===
geo_ts_mean              47.7499
road_type_filled         12.7125
geo_hour_mean            10.6395
road_type_ord             4.2122
lanes_x_road              4.0301
rt_ts_count               2.8634
prev_day_ts_demand        2.6468
rt_hour_mean              2.1728
rt_ts_mean                2.0236
large_veh_bin             1.9227
NumberofLanes             1.8815
rt_ts_confidence          1.6945
geo_mean                  1.4995
geo_ts_delta              0.8164
hour                      0.5157
ts_min                    0.4814
geo_hour_count            0.3477
geo_std                   0.3002
geo_count                 0.2926
geo_ts_count              0.2907
geo5_freq                 0.2785
demand_ratio_gh_hour      0.1442
geo4_freq                 0.1248
rt_missing                0.1031
time_slot_demand_mean     0.0822
sin_hour                  0.0740
geo4_mean                 0.0425
geo4                      0.0254
ts_mean              

In [14]:
# ════════════════════════════════════════════════════════════════════════
# FULL-DATASET TRAINING & SUBMISSION GENERATION
# ════════════════════════════════════════════════════════════════════════
print('Building full-train features (ref = full train)...')
full_fe = build_features(train, train)
y_full  = full_fe['demand']
print(f'Full train shape: {full_fe.shape}')

print('Building test features (ref = full train)...')
test_fe = build_features(test, train)
print(f'Test shape      : {test_fe.shape}')

# ── Helper: pick number of full-train iterations ─────────────────────────────
def _full_iters(val_model, factor=1.10, minimum=200):
    n = getattr(val_model, 'best_iteration_', None)
    if n is None or n <= 0:
        n = getattr(val_model, 'tree_count_', 1000)
    return max(minimum, int(n * factor))

_BASE_FULL_PARAMS = CAT_BASE_PARAMS.copy()
_BASE_FULL_PARAMS.pop('early_stopping_rounds', None)
_BASE_FULL_PARAMS.pop('eval_metric', None)
_BASE_FULL_PARAMS.update({'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05})


def _train_catboost_full(feats, cat_feats, n_iters, verbose=100):
    p = _BASE_FULL_PARAMS.copy()
    p['iterations'] = n_iters
    m = CatBoostRegressor(
        **p,
        cat_features=[c for c in cat_feats if c in feats],
        verbose=verbose,
    )
    m.fit(full_fe[feats], y_full)
    return m


# ────────────────────────────────────────────────────────────────────────────
# SUBMISSION A: submission_prev_day.csv
# CatBoost — BASE_FEATURES + prev_day_ts_demand only
# ────────────────────────────────────────────────────────────────────────────
print()
print('=== SUBMISSION A: CatBoost + prev_day_ts_demand only ===')

_v6_exclude_A = {'time_slot_demand_mean', 'demand_ratio_gh_hour'}
_feats_A = [f for f in BASE_FEATURES
            if f not in _v6_exclude_A and f in full_fe.columns]
_cats_A  = [c for c in BASE_CAT_FEATURES if c in _feats_A]

_iters_A = _full_iters(cat_model_v6)
model_A  = _train_catboost_full(_feats_A, _cats_A, _iters_A)
preds_A  = np.clip(model_A.predict(test_fe[_feats_A]), 0.0, 1.0)
sub_A    = pd.DataFrame({'Index': test['Index'], 'demand': preds_A})
sub_A.to_csv('submission_prev_day.csv', index=False)

print(f'submission_prev_day.csv       | shape={sub_A.shape} '
      f'| min={preds_A.min():.5f} | max={preds_A.max():.5f} '
      f'| mean={preds_A.mean():.5f} | nulls={sub_A["demand"].isna().sum()}')


# ────────────────────────────────────────────────────────────────────────────
# SUBMISSION B: submission_prev_day_ratio.csv
# CatBoost — BASE_FEATURES + prev_day_ts_demand + demand_ratio_gh_hour
# ────────────────────────────────────────────────────────────────────────────
print()
print('=== SUBMISSION B: CatBoost + prev_day + demand_ratio_gh_hour ===')

_v6_exclude_B = {'time_slot_demand_mean'}
_feats_B = [f for f in BASE_FEATURES
            if f not in _v6_exclude_B and f in full_fe.columns]
_cats_B  = [c for c in BASE_CAT_FEATURES if c in _feats_B]

model_B  = _train_catboost_full(_feats_B, _cats_B, _iters_A)
preds_B  = np.clip(model_B.predict(test_fe[_feats_B]), 0.0, 1.0)
sub_B    = pd.DataFrame({'Index': test['Index'], 'demand': preds_B})
sub_B.to_csv('submission_prev_day_ratio.csv', index=False)

print(f'submission_prev_day_ratio.csv | shape={sub_B.shape} '
      f'| min={preds_B.min():.5f} | max={preds_B.max():.5f} '
      f'| mean={preds_B.mean():.5f} | nulls={sub_B["demand"].isna().sum()}')


# ────────────────────────────────────────────────────────────────────────────
# SUBMISSION C: submission_v6_catboost.csv
# CatBoost — full final_features_v6 (all experiments kept)
# ────────────────────────────────────────────────────────────────────────────
print()
print('=== SUBMISSION C: CatBoost v6 (all kept features) ===')

_feats_C = [f for f in final_features_v6 if f in full_fe.columns]
_cats_C  = [c for c in final_cat_features_v6 if c in _feats_C]

model_C  = _train_catboost_full(_feats_C, _cats_C, _iters_A)
preds_C  = np.clip(model_C.predict(test_fe[_feats_C]), 0.0, 1.0)
sub_C    = pd.DataFrame({'Index': test['Index'], 'demand': preds_C})
sub_C.to_csv('submission_v6_catboost.csv', index=False)

print(f'submission_v6_catboost.csv    | shape={sub_C.shape} '
      f'| min={preds_C.min():.5f} | max={preds_C.max():.5f} '
      f'| mean={preds_C.mean():.5f} | nulls={sub_C["demand"].isna().sum()}')


# ────────────────────────────────────────────────────────────────────────────
# SUBMISSION D: submission_v6_blend.csv
# CatBoost v6 + LightGBM blend, weights from best_alpha found in Step 5
# ────────────────────────────────────────────────────────────────────────────
print()
print('=== SUBMISSION D: CatBoost v6 + LightGBM blend ===')
print(f'Blend alpha (CatBoost weight): {best_alpha:.2f}')

# Retrain LightGBM on full train with same n_iters scaling
full_lgb_tr, test_lgb_te, _, _ = _encode_for_lgb(full_fe, test_fe, final_features_v6)

dtrain_full_lgb = lgb.Dataset(full_lgb_tr[final_features_v6], label=y_full)
lgb_full_model  = lgb.train(
    _LGB_PARAMS,
    dtrain_full_lgb,
    num_boost_round=int(lgb_model_val.best_iteration * 1.10),
)

lgb_preds_test = np.clip(
    lgb_full_model.predict(test_lgb_te[final_features_v6]), 0.0, 1.0
)
preds_D = np.clip(
    best_alpha * preds_C + (1 - best_alpha) * lgb_preds_test, 0.0, 1.0
)
sub_D = pd.DataFrame({'Index': test['Index'], 'demand': preds_D})
sub_D.to_csv('submission_v6_blend.csv', index=False)

print(f'submission_v6_blend.csv       | shape={sub_D.shape} '
      f'| min={preds_D.min():.5f} | max={preds_D.max():.5f} '
      f'| mean={preds_D.mean():.5f} | nulls={sub_D["demand"].isna().sum()}')


# ────────────────────────────────────────────────────────────────────────────
# SUBMISSION AUDIT — Assert correctness of all 4 files
# ────────────────────────────────────────────────────────────────────────────
print()
print('=== SUBMISSION AUDIT ===')
_submissions = [
    ('submission_prev_day.csv',       sub_A, preds_A),
    ('submission_prev_day_ratio.csv', sub_B, preds_B),
    ('submission_v6_catboost.csv',    sub_C, preds_C),
    ('submission_v6_blend.csv',       sub_D, preds_D),
]
for fname, sub, preds in _submissions:
    assert sub.shape == (41778, 2), \
        f'SHAPE MISMATCH in {fname}: expected (41778, 2), got {sub.shape}'
    assert sub['demand'].isna().sum() == 0, \
        f'NULL VALUES in {fname}: {sub["demand"].isna().sum()} nulls'
    assert preds.min() >= 0.0, \
        f'NEGATIVE PREDICTIONS in {fname}: min={preds.min()}'
    assert preds.max() <= 1.0, \
        f'OUT-OF-RANGE PREDICTIONS in {fname}: max={preds.max()}'
    print(f'  ✓  {fname:<40}  shape={sub.shape}  '
          f'min={preds.min():.5f}  max={preds.max():.5f}  '
          f'mean={preds.mean():.5f}  nulls=0')
print()
print('All submissions passed audit.')

Building full-train features (ref = full train)...


Full train shape: (77299, 72)
Building test features (ref = full train)...


Test shape      : (41778, 71)

=== SUBMISSION A: CatBoost + prev_day_ts_demand only ===
0:	learn: 0.1356660	total: 13.5ms	remaining: 2.69s


100:	learn: 0.0156600	total: 1.08s	remaining: 1.06s


199:	learn: 0.0131318	total: 2.21s	remaining: 0us
submission_prev_day.csv       | shape=(41778, 2) | min=0.00195 | max=1.00000 | mean=0.11227 | nulls=0

=== SUBMISSION B: CatBoost + prev_day + demand_ratio_gh_hour ===
0:	learn: 0.1354861	total: 13.2ms	remaining: 2.63s


100:	learn: 0.0154795	total: 1.11s	remaining: 1.09s


199:	learn: 0.0130157	total: 2.21s	remaining: 0us
submission_prev_day_ratio.csv | shape=(41778, 2) | min=0.00228 | max=1.00000 | mean=0.11205 | nulls=0

=== SUBMISSION C: CatBoost v6 (all kept features) ===
0:	learn: 0.1355600	total: 12.6ms	remaining: 2.52s


100:	learn: 0.0154080	total: 1.23s	remaining: 1.21s


199:	learn: 0.0128534	total: 2.37s	remaining: 0us
submission_v6_catboost.csv    | shape=(41778, 2) | min=0.00135 | max=1.00000 | mean=0.11179 | nulls=0

=== SUBMISSION D: CatBoost v6 + LightGBM blend ===
Blend alpha (CatBoost weight): 0.00


submission_v6_blend.csv       | shape=(41778, 2) | min=0.00000 | max=1.00000 | mean=0.11305 | nulls=0

=== SUBMISSION AUDIT ===
  ✓  submission_prev_day.csv                   shape=(41778, 2)  min=0.00195  max=1.00000  mean=0.11227  nulls=0
  ✓  submission_prev_day_ratio.csv             shape=(41778, 2)  min=0.00228  max=1.00000  mean=0.11205  nulls=0
  ✓  submission_v6_catboost.csv                shape=(41778, 2)  min=0.00135  max=1.00000  mean=0.11179  nulls=0
  ✓  submission_v6_blend.csv                   shape=(41778, 2)  min=0.00000  max=1.00000  mean=0.11305  nulls=0

All submissions passed audit.


## 10. Summary

The executed cells above print:

| Item | Output |
|------|--------|
| Diagnostic report | Phase 1 diagnostics cell output |
| Count-feature results | Feature experiment results |
| Seen/unseen results | Feature experiment results |
| Smoothing results | Smoothing table |
| Hyperparameter results | Micro-tuning table |
| Ensemble results | Single-model and blend tables |
| Best validation R2 | Final summary cell |
| Final feature list | Final feature list table |
| Top feature importances | Top 30 table and targeted review |
| Validation design | Timestamp-aligned day-48 holdout, unchanged |
| Submission | `submission.csv` from the best temporal-validation strategy |

In [15]:
# ════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════
print('=' * 72)
print('FLIPKART GRIDLOCK v6 — FINAL SUMMARY')
print('=' * 72)
print()

_summary = [
    ('CatBoost v2 baseline',              0.664431),
    ('CatBoost v6 (all 3 new features)',  cat_r2_v6),
    ('LightGBM (same feature set)',       lgb_r2_val),
    ('Best CatBoost+LGB blend',           best_blend_r2),
]
print(f'{"Model/Config":<45} {"Val R2":>10} {"Est. LB Score":>14}')
print('-' * 72)
for label, r2 in _summary:
    print(f'{label:<45} {r2:>10.6f} {max(0, r2*100):>14.4f}')
print()

print('=== EXPERIMENT TRACKING TABLE ===')
print(pd.DataFrame(experiment_log).to_string(index=False))

print()
print('=== SUBMISSION RECOMMENDATION ===')
_sub_scores = {
    'submission_prev_day.csv       (CatBoost+prev_day_only)' : 0.0,  # val R2 not tracked separately
    'submission_prev_day_ratio.csv (CatBoost+prev+ratio)'    : 0.0,
    'submission_v6_catboost.csv    (CatBoost v6 full)'        : cat_r2_v6,
    'submission_v6_blend.csv       (CB+LGB blend)'            : best_blend_r2,
}
best_sub_label = max(_sub_scores, key=_sub_scores.get)
print(f'HIGHEST VAL R2: {best_sub_label}')
print(f'Val R2 = {_sub_scores[best_sub_label]:.6f}')
print()
print('CRITICAL INTERPRETATION:')
print('  Validation measures FALLBACK behavior (0% exact geo×ts coverage).')
print('  Leaderboard measures EXACT geo×ts lookup (88.9% test coverage).')
print('  The best leaderboard submission may NOT be the highest validation R2.')
print()
print('RECOMMENDED SUBMISSION ORDER:')
print('  1. submission_v6_catboost.csv  — primary bet (full v6, highest coverage signal)')
print('  2. submission_v6_blend.csv     — hedge (blend helps fallback rows: 11.1% of test)')
print('  3. submission_prev_day.csv     — diagnostic (isolates prev_day signal impact)')
print('  4. submission_prev_day_ratio.csv — diagnostic (adds ratio effect)')
print()
print('EXPECTED GAP TO FRIEND:')
print(f'  Current LB: 85.23724  →  Friend LB: 89.82959  →  Gap: 4.59 pts')
print(f'  prev_day_ts_demand covers {raw_test.notna().mean():.1%} of test rows at day-48 precision.')
print(f'  This is the exact signal type behind the friend gap. Submit and verify.')

FLIPKART GRIDLOCK v6 — FINAL SUMMARY

Model/Config                                      Val R2  Est. LB Score
------------------------------------------------------------------------
CatBoost v2 baseline                            0.664431        66.4431
CatBoost v6 (all 3 new features)                0.603607        60.3607
LightGBM (same feature set)                     0.617472        61.7472
Best CatBoost+LGB blend                         0.617472        61.7472

=== EXPERIMENT TRACKING TABLE ===
                  Feature Added                               Coverage  Validation R2 Before  Validation R2 After      Gain Keep?
             prev_day_ts_demand                   Val=0.0%  Test=88.9%              0.611772             0.603607 -0.008165    NO
           demand_ratio_gh_hour 100% (computed from existing features)              0.611772             0.603607 -0.008165    NO
          time_slot_demand_mean        100% (all 96 slots in training)              0.611772            

In [16]:

# ════════════════════════════════════════════════════════════════════════
# OOF TARGET ENCODING EXPERIMENTS (PHASE 1)
# ════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

print("Starting OOF Target Encoding Phase...")

def generate_oof_features(base_df, target_df, cols_to_encode, n_splits=5, seed=42):
    """
    Generate OOF target encodings.
    For base_df (training data), uses 5-fold CV to compute out-of-fold means.
    For target_df (validation/test data), uses the full base_df to compute means.
    Returns copies of base_df and target_df with new features appended.
    """
    base_encoded = base_df.copy()
    target_encoded = target_df.copy()
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = base_df['demand'].mean()
    
    for cols in cols_to_encode:
        if isinstance(cols, str):
            cols = [cols]
        
        feature_name = "_".join(cols) + "_te"
        base_encoded[feature_name] = np.nan
        
        # OOF for base_df
        for tr_idx, val_idx in kf.split(base_df):
            tr_fold = base_df.iloc[tr_idx]
            val_fold = base_df.iloc[val_idx]
            
            grp_mean = tr_fold.groupby(cols)['demand'].mean()
            
            if len(cols) == 1:
                base_encoded.loc[base_encoded.index[val_idx], feature_name] = val_fold[cols[0]].map(grp_mean)
            else:
                # Map across multiple columns
                val_keys = val_fold.set_index(cols).index
                grp_mean_dict = grp_mean.to_dict()
                base_encoded.loc[base_encoded.index[val_idx], feature_name] = val_keys.map(lambda x: grp_mean_dict.get(x, np.nan))
                
        # Fill missing OOF with global mean
        base_encoded[feature_name] = base_encoded[feature_name].fillna(global_mean)
        
        # Full map for target_df
        full_grp_mean = base_df.groupby(cols)['demand'].mean()
        if len(cols) == 1:
            target_encoded[feature_name] = target_df[cols[0]].map(full_grp_mean)
        else:
            target_keys = target_df.set_index(cols).index
            full_grp_mean_dict = full_grp_mean.to_dict()
            target_encoded[feature_name] = target_keys.map(lambda x: full_grp_mean_dict.get(x, np.nan))
            
        target_encoded[feature_name] = target_encoded[feature_name].fillna(global_mean)
        
    return base_encoded, target_encoded

# Define features to encode
tier_1 = ['geohash', ['geohash', 'hour'], ['geohash', 'timestamp']]
tier_2 = ['road_type_filled', ['road_type_filled', 'hour'], ['road_type_filled', 'timestamp']]
tier_3 = ['weather_filled', ['weather_filled', 'hour'], 'landmark_bin']

all_te_configs = tier_1 + tier_2 + tier_3
all_te_names = ["_".join(c) + "_te" if isinstance(c, list) else c + "_te" for c in all_te_configs]

# ── 1. Generate OOF Features for Validation Split ───────────────────────
print("Generating OOF features for Validation Split (tr_df -> val_df)...")
tr_fe_oof, val_fe_oof = generate_oof_features(tr_fe, val_fe, all_te_configs)

# ── 2. Generate OOF Features for Full Train Split ───────────────────────
print("Generating OOF features for Full Train Split (train -> test)...")
full_fe_oof, test_fe_oof = generate_oof_features(full_fe, test_fe, all_te_configs)


# ── EXPERIMENT TRACKER ──────────────────────────────────────────────────
baseline_features = final_features_v6.copy()
baseline_cat_features = final_cat_features_v6.copy()
baseline_r2 = cat_r2_v6
best_oof_r2 = baseline_r2
best_oof_features = baseline_features.copy()

print(f"\n{'='*60}\nBASELINE R²: {baseline_r2:.6f}\n{'='*60}")

def run_oof_experiment(exp_name, base_features, added_features):
    candidate_features = base_features + added_features
    # We do NOT add target encodings to cat_features! They are continuous floats.
    p = CAT_BASE_PARAMS.copy()
    p['iterations'] = int(p['iterations'] * 1.5) if 'iterations' in p else 2000 # Enough for tuning
    
    m = CatBoostRegressor(**p, cat_features=baseline_cat_features, verbose=0)
    m.fit(tr_fe_oof[candidate_features], y_tr, eval_set=(val_fe_oof[candidate_features], y_val), early_stopping_rounds=100)
    
    preds = np.clip(m.predict(val_fe_oof[candidate_features]), 0.0, 1.0)
    score = r2_score(y_val, preds)
    gain = score - baseline_r2
    keep = score > best_oof_r2
    
    print(f"\n=== {exp_name} ===")
    print(f"Features added       : {added_features}")
    print(f"Validation R² Before : {baseline_r2:.6f}")
    print(f"Validation R² After  : {score:.6f}")
    print(f"Gain                 : {gain:+.6f}")
    print(f"Decision             : {'Keep' if keep else 'Drop'}")
    
    # Feature Importance
    fi = pd.DataFrame({'feature': candidate_features, 'importance': m.feature_importances_})
    fi = fi.sort_values('importance', ascending=False)
    
    print("\nSpecific Importance of OOF features:")
    oof_fi = fi[fi['feature'].isin(added_features)]
    if not oof_fi.empty:
        print(oof_fi.to_string(index=False))
    
    print("\nTop 20 Features Overall:")
    print(fi.head(20).to_string(index=False))
    
    return score, m, keep

# ── EXPERIMENT A: Baseline + geohash_te ───────────────────────────────
score_a, model_a, keep_a = run_oof_experiment("Experiment A", baseline_features, ['geohash_te'])
if keep_a: best_oof_r2, best_oof_features = score_a, baseline_features + ['geohash_te']

# ── EXPERIMENT B: Exp A + geohash_hour_te ─────────────────────────────
score_b, model_b, keep_b = run_oof_experiment("Experiment B", best_oof_features, ['geohash_hour_te'])
if keep_b: best_oof_r2, best_oof_features = score_b, best_oof_features + ['geohash_hour_te']

# ── EXPERIMENT C: Exp B + geohash_timestamp_te ────────────────────────
score_c, model_c, keep_c = run_oof_experiment("Experiment C", best_oof_features, ['geohash_timestamp_te'])
if keep_c: best_oof_r2, best_oof_features = score_c, best_oof_features + ['geohash_timestamp_te']

# ── EXPERIMENT D: Best previous + roadtype encodings ──────────────────
score_d, model_d, keep_d = run_oof_experiment("Experiment D", best_oof_features, ['road_type_filled_te', 'road_type_filled_hour_te', 'road_type_filled_timestamp_te'])
if keep_d: best_oof_r2, best_oof_features = score_d, best_oof_features + ['road_type_filled_te', 'road_type_filled_hour_te', 'road_type_filled_timestamp_te']

# ── EXPERIMENT E: Best previous + remaining encodings ─────────────────
score_e, model_e, keep_e = run_oof_experiment("Experiment E", best_oof_features, ['weather_filled_te', 'weather_filled_hour_te', 'landmark_bin_te'])
if keep_e: best_oof_r2, best_oof_features = score_e, best_oof_features + ['weather_filled_te', 'weather_filled_hour_te', 'landmark_bin_te']

print(f"\n{'='*60}\nBEST FINAL OOF R²: {best_oof_r2:.6f} (Gain: {best_oof_r2 - baseline_r2:+.6f})\n{'='*60}")
print(f"Final OOF Features kept: {[f for f in best_oof_features if f not in baseline_features]}")


# ── AUDIT SECTION ───────────────────────────────────────────────────────
print("\n" + "="*60 + "\nOOF FEATURE AUDIT\n" + "="*60)
audit_rows = []
global_mean = tr_fe['demand'].mean()

for te_col in all_te_names:
    val_data = val_fe_oof[te_col]
    tr_data = tr_fe_oof[te_col]
    
    # Missing / fallback
    missing_mask = np.isclose(val_data, global_mean)
    missing_pct = missing_mask.mean() * 100
    coverage = 100 - missing_pct
    
    # Correlation
    corr, _ = pearsonr(val_data, y_val)
    
    # Single-feature addition R²
    p = CAT_BASE_PARAMS.copy()
    p['iterations'] = 500
    m = CatBoostRegressor(**p, cat_features=baseline_cat_features, verbose=0)
    m.fit(tr_fe_oof[baseline_features + [te_col]], y_tr, eval_set=(val_fe_oof[baseline_features + [te_col]], y_val), early_stopping_rounds=50)
    preds = np.clip(m.predict(val_fe_oof[baseline_features + [te_col]]), 0.0, 1.0)
    r2 = r2_score(y_val, preds)
    
    fi_val = m.feature_importances_[m.feature_names_.index(te_col)]
    
    audit_rows.append({
        'Feature': te_col,
        'Coverage (%)': f"{coverage:.2f}",
        'Missing (%)': f"{missing_pct:.2f}",
        'Correlation': f"{corr:.4f}",
        'Importance': f"{fi_val:.2f}",
        'Val R²': f"{r2:.6f}",
        'Val Gain': f"{r2 - baseline_r2:+.6f}"
    })

audit_df = pd.DataFrame(audit_rows)
print(audit_df.to_string(index=False))


# ── SUBMISSION GENERATION ───────────────────────────────────────────────
print("\n" + "="*60 + "\nGENERATING SUBMISSIONS\n" + "="*60)

def train_and_predict_oof(exp_features, filename):
    iters = _full_iters(cat_model_v6)
    p = _BASE_FULL_PARAMS.copy()
    p['iterations'] = iters
    model = CatBoostRegressor(**p, cat_features=baseline_cat_features, verbose=0)
    model.fit(full_fe_oof[exp_features], y_full)
    
    preds = np.clip(model.predict(test_fe_oof[exp_features]), 0.0, 1.0)
    sub = pd.DataFrame({'Index': test['Index'], 'demand': preds})
    sub.to_csv(filename, index=False)
    print(f"Saved {filename} | Mean: {preds.mean():.5f}")

# Experiment A
train_and_predict_oof(baseline_features + ['geohash_te'], 'submission_oof_geo.csv')
# Experiment B
train_and_predict_oof(baseline_features + ['geohash_te', 'geohash_hour_te'], 'submission_oof_geo_hour.csv')
# Experiment C
train_and_predict_oof(baseline_features + ['geohash_te', 'geohash_hour_te', 'geohash_timestamp_te'], 'submission_oof_geo_ts.csv')
# Final Full
train_and_predict_oof(best_oof_features, 'submission_oof_full.csv')

print("OOF Phase Complete.")



Starting OOF Target Encoding Phase...
Generating OOF features for Validation Split (tr_df -> val_df)...


Generating OOF features for Full Train Split (train -> test)...



BASELINE R²: 0.603607



=== Experiment A ===
Features added       : ['geohash_te']
Validation R² Before : 0.603607
Validation R² After  : 0.503531
Gain                 : -0.100076
Decision             : Drop

Specific Importance of OOF features:
   feature  importance
geohash_te    0.136016

Top 20 Features Overall:
           feature  importance
       geo_ts_mean   43.910108
     geo_hour_mean   14.781082
        rt_ts_mean    8.301092
      rt_hour_mean    5.953826
  rt_ts_confidence    4.787380
       rt_ts_count    4.719431
     road_type_ord    4.319003
  road_type_filled    3.008611
     NumberofLanes    2.797866
prev_day_ts_demand    2.035080
      lanes_x_road    1.202166
     large_veh_bin    0.807210
          geo_mean    0.382719
              hour    0.318068
            ts_min    0.316893
      geo_ts_count    0.314462
      geo_ts_delta    0.284083
         geo4_mean    0.283414
    geo_hour_count    0.279458
           geo_std    0.167053



=== Experiment B ===
Features added       : ['geohash_hour_te']
Validation R² Before : 0.603607
Validation R² After  : 0.501511
Gain                 : -0.102096
Decision             : Drop

Specific Importance of OOF features:
        feature  importance
geohash_hour_te    0.398022

Top 20 Features Overall:
           feature  importance
       geo_ts_mean   44.120146
     geo_hour_mean   14.880078
        rt_ts_mean    8.635171
      rt_hour_mean    5.899103
  rt_ts_confidence    4.838466
       rt_ts_count    4.458355
     road_type_ord    4.423414
  road_type_filled    3.138720
     NumberofLanes    2.773709
prev_day_ts_demand    1.445386
      lanes_x_road    1.232533
     large_veh_bin    0.725667
   geohash_hour_te    0.398022
            ts_min    0.388766
          geo_mean    0.385909
      geo_ts_delta    0.312615
    geo_hour_count    0.305519
              hour    0.204113
           geo_std    0.202750
          sin_hour    0.186417


...

                      Feature Coverage (%) Missing (%) Correlation Importance   Val R²  Val Gain
                   geohash_te        99.21        0.79      0.8563       0.14 0.503531 -0.100076
              geohash_hour_te         6.11       93.89      0.1699       0.40 0.501511 -0.102096
         geohash_timestamp_te         0.00      100.00         nan       6.50 0.503233 -0.100374
          road_type_filled_te       100.00        0.00      0.8699       8.08 0.602929 -0.000678
     road_type_filled_hour_te         6.40       93.60      0.1555       4.98 0.491299 -0.112308
road_type_filled_timestamp_te         0.00      100.00         nan       7.37 0.511654 -0.091953
            weather_filled_te       100.00        0.00     -0.0019       0.01 0.498404 -0.105203
       weather_filled_hour_te         6.40       93.60     -0.0344       0.01 0.501922 -0.101685
              landmark_bin_te       100.00        0.00      0.0194       0.00 0.495358 -0.108249

GENERATING SUBMISSIONS


Saved submission_oof_geo.csv | Mean: 0.11234


Saved submission_oof_geo_hour.csv | Mean: 0.11216


Saved submission_oof_geo_ts.csv | Mean: 0.11071


Saved submission_oof_full.csv | Mean: 0.11217
OOF Phase Complete.
